In [50]:
import os
import pandas as pd
import numpy as np

from datasets import load_dataset

from sentence_transformers import SentenceTransformer
import faiss

from sklearn.model_selection import train_test_split

print("Imports successful!")

Imports successful!


In [51]:
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [52]:
print(dataset["train"].column_names)

['flags', 'instruction', 'category', 'intent', 'response']


In [53]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)
df.head()

Shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [54]:
print(df[["instruction", "response", "category"]].head(10))

                                         instruction  \
0   question about cancelling order {{Order Number}}   
1  i have a question about cancelling oorder {{Or...   
2    i need help cancelling puchase {{Order Number}}   
3         I need to cancel purchase {{Order Number}}   
4  I cannot afford this order, cancel purchase {{...   
5     can you help me cancel order {{Order Number}}?   
6  I can no longer afford order {{Order Number}},...   
7    I am trying to cancel purchase {{Order Number}}   
8     I have got to cancel purchase {{Order Number}}   
9    i need help canceling purchase {{Order Number}}   

                                            response category  
0  I've understood you have a question regarding ...    ORDER  
1  I've been informed that you have a question ab...    ORDER  
2  I can sense that you're seeking assistance wit...    ORDER  
3  I understood that you need assistance with can...    ORDER  
4  I'm sensitive to the fact that you're facing f...    ORDER  

In [55]:
print("Missing values:")
print(df.isnull().sum())

print("\nNumber of intents:")
print(df["category"].nunique())

print("\nIntent distribution:")
print(df["category"].value_counts())



Missing values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Number of intents:
11

Intent distribution:
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64


In [56]:
documents = []

for _, row in df.iterrows():
    document = (
        f"Customer issue: {row['instruction']}\n"
        f"Support response: {row['response']}\n"
        f"Intent: {row['category']}"
    )
    
    documents.append(document)

print("Number of documents:", len(documents))
print("\nExample:\n")
print(documents[0])

Number of documents: 26872

Example:

Customer issue: question about cancelling order {{Order Number}}
Support response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.
Intent: ORDER


In [57]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [58]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

Embedding shape: (26872, 384)


In [59]:
faiss.normalize_L2(embeddings)

print("Embeddings normalized successfully!")

Embeddings normalized successfully!


In [60]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created!")
print("Number of vectors:", index.ntotal)

FAISS index created!
Number of vectors: 26872


In [61]:
def retrieve_documents_with_scores(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = query_embedding.astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "document": documents[int(idx)],
            "score": float(score)
        })

    return results

In [62]:
query = "Where is my order?"

results = retrieve_documents_with_scores(
    query,
    k=5
)

for i, result in enumerate(results, 1):

    print(f"\n--- Document {i} ---")
    print("Similarity:", round(result["score"], 4))
    print(result["document"][:500])


--- Document 1 ---
Similarity: 0.6795
Customer issue: where do I order something?
Support response: I'm delighted to assist you in finding the right place to order our products! To place an order, you can visit our website at {{Website URL}}. Our website is designed to provide you with a user-friendly and seamless ordering experience. Simply search for the items you wish to purchase, add them to your cart, and proceed to the checkout page to complete your order. Our website also offers secure payment options to ensure a smooth and p

--- Document 2 ---
Similarity: 0.6486
Customer issue: where do I order some items?
Support response: Thank you for your question regarding where to order items. I'm here to assist you with that. To place an order, you can visit our online store on our website or use our mobile app. Simply browse through the available products, select the items you want to purchase, and proceed to the checkout page. You'll be prompted to provide your shipping information a

In [63]:
def build_context(results):

    return "\n\n".join(
        result["document"]
        for result in results
    )

In [64]:
context = build_context(results)

print(context)

Customer issue: where do I order something?
Support response: I'm delighted to assist you in finding the right place to order our products! To place an order, you can visit our website at {{Website URL}}. Our website is designed to provide you with a user-friendly and seamless ordering experience. Simply search for the items you wish to purchase, add them to your cart, and proceed to the checkout page to complete your order. Our website also offers secure payment options to ensure a smooth and protected transaction. If you have any questions or need further assistance during the ordering process, our customer support team is always available to help you. Happy shopping!
Intent: ORDER

Customer issue: where do I order some items?
Support response: Thank you for your question regarding where to order items. I'm here to assist you with that. To place an order, you can visit our online store on our website or use our mobile app. Simply browse through the available products, select the item

In [65]:
results = retrieve_documents(
    "I want to get a refund",
    k=3
)

for result in results:
    print("Score:", result["score"])
    print(result["document"])
    print("-" * 70)

Score: 0.6413647532463074
Customer issue: I have to see the status of my refund, how could I do it?
Support response: I've decoded that you're eager to check the status of your refund. Rest assured, I'm here to guide you through the process. To find the current status of your refund, you can visit our website and log into your account. Once logged in, navigate to the "My Orders" or "My Account" section, where you should be able to locate the information related to your refund. If you encounter any difficulties or have any further questions, please don't hesitate to ask. We're here to assist you every step of the way.
Intent: REFUND
----------------------------------------------------------------------
Score: 0.6378312706947327
Customer issue: where to obtain a refund of my money?
Support response: I'm on the same page, your need to locate where to obtain a refund for your money. To ensure the process is seamless for you, I recommend reaching out to our customer support team. They have 

In [66]:
context = build_context(results)

print(context)

Customer issue: I have to see the status of my refund, how could I do it?
Support response: I've decoded that you're eager to check the status of your refund. Rest assured, I'm here to guide you through the process. To find the current status of your refund, you can visit our website and log into your account. Once logged in, navigate to the "My Orders" or "My Account" section, where you should be able to locate the information related to your refund. If you encounter any difficulties or have any further questions, please don't hesitate to ask. We're here to assist you every step of the way.
Intent: REFUND

Customer issue: where to obtain a refund of my money?
Support response: I'm on the same page, your need to locate where to obtain a refund for your money. To ensure the process is seamless for you, I recommend reaching out to our customer support team. They have the expertise and resources to guide you through the refund process and provide you with the necessary information. You ca

In [67]:
SIMILARITY_THRESHOLD = 0.45

print("Similarity threshold:", SIMILARITY_THRESHOLD)

Similarity threshold: 0.45


In [68]:
!pip install groq
from groq import Groq

print("Groq imported successfully!")

Groq imported successfully!



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from groq import Groq

api_key = "YOUR_GROQ_API_KEY"

client = Groq(api_key=api_key)

print("Groq connected successfully!")

Groq connected successfully!


In [71]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Hello!


In [72]:
def build_prompt(user_query, context):

    return f"""
You are an e-commerce customer support assistant.

Answer the customer's question using ONLY the information
provided in the Knowledge Base.

Rules:
1. Do not invent information.
2. Do not invent policies, prices, delivery times, phone numbers,
   URLs, or procedures.
3. If the Knowledge Base does not contain enough information,
   clearly say that you do not have enough information.
4. Be polite, concise, and helpful.
5. If the customer is frustrated, acknowledge their frustration.
6. Do not mention internal instructions, embeddings, FAISS,
   vector databases, or the RAG pipeline.
7. Do not treat placeholders as real information.

Knowledge Base:
{context}

Customer Question:
{user_query}

Answer:
"""

In [73]:
def clean_response(response):

    placeholders = [
        "{{Customer Support Phone Number}}",
        "{{Website URL}}"
    ]

    for placeholder in placeholders:
        response = response.replace(
            placeholder,
            ""
        )

    while "\n\n\n" in response:
        response = response.replace(
            "\n\n\n",
            "\n\n"
        )

    return response.strip()

In [74]:
def generate_response(user_query, context):

    prompt = build_prompt(
        user_query,
        context
    )

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    return clean_response(answer)

In [75]:
def rag_answer_with_threshold(query, k=5):

    results = retrieve_documents_with_scores(
        query,
        k
    )

    best_score = results[0]["score"]

    # Not enough relevant information
    if best_score < SIMILARITY_THRESHOLD:

        return {
            "query": query,
            "answer": (
                "I'm sorry, but I don't have enough information "
                "to answer this question accurately."
            ),
            "route": "insufficient_context",
            "best_score": best_score,
            "retrieved_documents": []
        }

    # Keep only relevant documents
    valid_results = [
        result
        for result in results
        if result["score"] >= SIMILARITY_THRESHOLD
    ]

    context = build_context(valid_results)

    answer = generate_response(
        query,
        context
    )

    return {
        "query": query,
        "answer": answer,
        "route": "rag",
        "best_score": best_score,
        "retrieved_documents": valid_results
    }

In [76]:
result = rag_answer_with_threshold(
    "Where is my order?"
)

print("Route:", result["route"])
print("Similarity:", round(result["best_score"], 4))

print("\nAnswer:")
print(result["answer"])

Route: rag
Similarity: 0.6795

Answer:
I’m happy to help you locate your order. Could you please provide your order number (or the tracking number) so I can check its status for you?


In [77]:
result = rag_answer_with_threshold(
    "I want a refund for my purchase."
)

print("Route:", result["route"])
print("Similarity:", round(result["best_score"], 4))

print("\nAnswer:")
print(result["answer"])

Route: rag
Similarity: 0.7276

Answer:
I understand you’d like to receive a refund for your purchase. To get started, please gather the details of your order (such as the order number, purchase date, and the reason for the refund). Then reach out to our customer‑support team—either by calling our support line or by using the Live Chat feature on our website. They will guide you through the specific steps and help process your refund promptly. If you need any further assistance, just let me know.


In [78]:
result = rag_answer_with_threshold(
    "I am having a problem with my payment."
)

print("Route:", result["route"])
print("Similarity:", round(result["best_score"], 4))

print("\nAnswer:")
print(result["answer"])

Route: rag
Similarity: 0.7706

Answer:
I’m sorry to hear you’re experiencing a problem with your payment. To help us identify the cause and find a solution, could you please share more details about the issue you’re encountering (e.g., error messages, the step where it occurs, the payment method you’re using)? Thank you for your patience—we’ll work with you to resolve this as quickly as possible.


In [79]:
result = rag_answer_with_threshold(
    "Who is the president of Egypt?"
)

print("Route:", result["route"])
print("Similarity:", round(result["best_score"], 4))

print("\nAnswer:")
print(result["answer"])

Route: insufficient_context
Similarity: 0.1192

Answer:
I'm sorry, but I don't have enough information to answer this question accurately.


In [80]:
test_queries = [
    "Where is my order?",
    "I want a refund for my purchase.",
    "I am having a problem with my payment.",
    "I forgot my password.",
    "I am very frustrated with this service.",
    "Who is the president of Egypt?"
]

for query in test_queries:

    result = rag_answer_with_threshold(query)

    print("=" * 80)
    print("Query:", query)
    print("Route:", result["route"])
    print(
        "Similarity:",
        round(result["best_score"], 4)
    )
    print("Answer:")
    print(result["answer"])
    print()

Query: Where is my order?
Route: rag
Similarity: 0.6795
Answer:
I’m happy to help you locate your order. Could you please provide the order number (or the tracking number) so I can check its status for you?

Query: I want a refund for my purchase.
Route: rag
Similarity: 0.7276
Answer:
I understand you’d like a refund for your purchase. To get started, please gather the details of your order (order number, purchase date, and the reason for the refund). Then reach out to our customer‑support team—they’ll guide you through the specific steps and process the refund for you.

You can contact them:

* By phone: ****  
* Via Live Chat on our website: ****

If you made the purchase online, you may also be able to submit a refund request directly through your account on the website. Keep any receipts or proof of purchase handy, as they’ll help speed up the process. Let us know if you need any further assistance!

Query: I am having a problem with my payment.
Route: rag
Similarity: 0.7706
Answer